# S47_04 — Retrieval Strategies

Dense vector search alone often misses exact keyword matches. Production RAG systems combine multiple retrieval signals.

## Dense vs Sparse retrieval

| Method | How it works | Strengths | Weaknesses |
|--------|-------------|-----------|------------|
| **Dense** (embeddings) | ANN search in vector space | Semantic similarity, paraphrases | Misses rare keywords, proper nouns |
| **Sparse** (BM25/TF-IDF) | Keyword frequency scoring | Exact term matching, interpretable | Misses synonyms, rephrasing |
| **Hybrid** | Combine both scores | Best of both worlds | More complex pipeline |

In [ ]:
# pip install rank-bm25 sentence-transformers
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np

corpus = [
    'Gradient descent is an optimization algorithm for machine learning.',
    'Adam optimizer combines momentum and RMSprop for adaptive learning rates.',
    'Backpropagation computes gradients through the chain rule.',
    'Learning rate scheduling reduces the learning rate during training.',
    'Batch normalization normalizes inputs to each layer to speed up training.',
    'Dropout is a regularization technique that randomly drops neurons.',
    'Transfer learning reuses pretrained model weights for new tasks.',
    'The PyTorch Adam optimizer implementation uses weight decay separately.',
]

# BM25 — sparse retrieval
tokenized_corpus = [doc.lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

query = 'Adam optimizer learning rate'
bm25_scores = bm25.get_scores(query.lower().split())

print('BM25 top results:')
for idx in np.argsort(bm25_scores)[::-1][:3]:
    print(f'  [{bm25_scores[idx]:.3f}] {corpus[idx]}')

In [ ]:
import faiss

embed_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
corpus_emb = embed_model.encode(corpus)
faiss.normalize_L2(corpus_emb)

d = corpus_emb.shape[1]
index = faiss.IndexFlatIP(d)
index.add(corpus_emb)

query_emb = embed_model.encode([query])
faiss.normalize_L2(query_emb)
dense_scores, dense_indices = index.search(query_emb, k=len(corpus))
dense_scores = dense_scores[0]
dense_indices = dense_indices[0]

# Reconstruct score array in original corpus order
dense_score_arr = np.zeros(len(corpus))
for score, idx in zip(dense_scores, dense_indices):
    dense_score_arr[idx] = score

print('\nDense top results:')
for idx in np.argsort(dense_score_arr)[::-1][:3]:
    print(f'  [{dense_score_arr[idx]:.3f}] {corpus[idx]}')

In [ ]:
# Hybrid retrieval — Reciprocal Rank Fusion (RRF)
# RRF score: sum of 1/(k + rank_i) across retrieval methods

def rrf_score(rank, k=60):
    return 1 / (k + rank)

def hybrid_retrieve(query, corpus, bm25, embed_model, index, top_k=3, k_rrf=60):
    n = len(corpus)
    
    # BM25 ranks
    bm25_scores = bm25.get_scores(query.lower().split())
    bm25_ranks = {idx: rank + 1 for rank, idx in enumerate(np.argsort(bm25_scores)[::-1])}
    
    # Dense ranks
    q_emb = embed_model.encode([query])
    faiss.normalize_L2(q_emb)
    _, d_idx = index.search(q_emb, k=n)
    dense_ranks = {idx: rank + 1 for rank, idx in enumerate(d_idx[0])}
    
    # RRF fusion
    rrf = {}
    for doc_idx in range(n):
        rrf[doc_idx] = rrf_score(bm25_ranks[doc_idx], k_rrf) + rrf_score(dense_ranks.get(doc_idx, n), k_rrf)
    
    ranked = sorted(rrf, key=rrf.get, reverse=True)
    return [(corpus[i], rrf[i]) for i in ranked[:top_k]]

results = hybrid_retrieve(query, corpus, bm25, embed_model, index)
print('Hybrid (RRF) top results:')
for text, score in results:
    print(f'  [{score:.4f}] {text}')

## Reranking — cross-encoder refinement

In [ ]:
# Cross-encoders jointly encode query + document for more accurate scoring
# Much slower than bi-encoders — use on a small candidate set (top-20 → rerank to top-5)

from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# First-stage: get top-5 candidates (any retrieval method)
candidates = [corpus[i] for i in np.argsort(dense_score_arr)[::-1][:5]]

# Second-stage: rerank with cross-encoder
pairs = [[query, doc] for doc in candidates]
rerank_scores = reranker.predict(pairs)

reranked = sorted(zip(rerank_scores, candidates), reverse=True)
print('Reranked results:')
for score, doc in reranked:
    print(f'  [{score:.3f}] {doc}')

## Query transformation

In [ ]:
import anthropic

client = anthropic.Anthropic()

def expand_query(query, n_variants=3):
    """Generate multiple query variants to improve recall."""
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        system='You are a search query expansion assistant. Generate alternative phrasings of a query to improve document retrieval. Respond with a JSON array of strings only.',
        messages=[{
            'role': 'user',
            'content': f'Generate {n_variants} alternative phrasings for this search query: "{query}"'
        }],
    )
    import json
    return json.loads(msg.content[0].text)

original_query = 'How do I stop my model from memorizing training data?'
variants = expand_query(original_query)
print(f'Original: {original_query}')
print('Variants:')
for v in variants:
    print(f'  - {v}')

## Retrieval pipeline summary

```
Query
  → [Query expansion / HyDE]
  → [Dense retrieval] + [Sparse BM25] → [RRF fusion]
  → [Reranking (cross-encoder)]
  → Top-K chunks → LLM
```

**HyDE** (Hypothetical Document Embeddings): generate a hypothetical answer, embed it, then retrieve — often improves recall for complex queries.

Next: [S47_05_end_to_end_rag.ipynb](./S47_05_end_to_end_rag.ipynb)